In [1]:
# LLM Setup
!apt-get update -y
!apt-get install -y zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

print("Ollama Started")

Ollama Started


In [3]:
!which ollama

/usr/local/bin/ollama


In [4]:
# Pull models
# 1 : LLM model = llama3
!ollama pull llama3

# 2 : Enmbeddings Gen-model = "mxbai-embed-large"
!ollama pull mxbai-embed-large

In [5]:
!pip install ollama
!pip install chromadb
!pip install pypdf
!pip install python-docx
!pip install numpy
!pip install sentence-transformers
!pip install tqdm

In [6]:
# Testing :
import ollama

response = ollama.chat(
    model = "llama3",
    messages = [{"role" : "user",
                 "content" : "Hello"}]
)

print(response['message']['content'])

Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [7]:
!ollama --version

ollama version is 0.17.6


In [12]:
import os
from google.colab import files
from pypdf import PdfReader
from docx import Document

# Upload File
def upload_file():
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    print(f"Uploaded File: {file_name}")
    return file_name


# Extract Text
def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text

    elif ext == ".txt":
        with open(file_path, encoding="utf-8") as f:
            return f.read()

    elif ext == ".docx":
        doc = Document(file_path)
        return "\n".join([pa.text for pa in doc.paragraphs])

    else:
        raise ValueError("Unsupported file type")

In [14]:
# Remove Nextline:
def clean_text(text):
  text = text.replace("\n", " ")
  text = text.replace("\t", " ")
  return text

**Create Chunks :-**

In [17]:
def chunk_text(text, chunk_size=500, overlap = 100):
  chunks = []
  start = 0
  text_len = len(text)

  while start < text_len :
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start = end - overlap

  return chunks


def process_documents():
  file_path = upload_file()
  raw_text = extract_text(file_path)
  cleaned_text = clean_text(raw_text)
  chunks = chunk_text(cleaned_text)
  return chunks

  print(f"\nTotal Chunks Created: {len(chunks)}")
  return chunks

chunks = process_documents()

print(f"\n--- Sample Chunks ---")
for i, chunk in enumerate(chunks[:2]):
  print(f"\nChunk {i+1}:\n{chunk[:300]}")

Saving Fundamental of Data Science.pdf to Fundamental of Data Science.pdf
Uploaded File: Fundamental of Data Science.pdf

--- Sample Chunks ---

Chunk 1:
-: Fundamental of Data Science :-    Q.1 :- Explain Data-Science with proper Diagram.  ➢ Ans:- Data Science :  - Data Science is a process to perform tasks on a Row-Data / Row-Things and manage the data  is Preferable way.  -OR-  - Data science is the process of turning messy, raw data—like numbers,

Chunk 2:
ons    - Data Science is an interdisciplinary field that uses scientific methods, processes, algorithms,  and systems to extract knowledge and insights from data.  It involves collecting, cleaning,  analyzing, and interpreting data to identify patterns, trends, and re lationships that can be  used t


**Generate Embeddings :-**

In [20]:
import ollama
import chromadb
from chromadb.utils import embedding_functions

In [21]:
# Create Collaction:
client= chromadb.Client()

collaction = client.create_collection(
    name = "doc_collection"
)

In [25]:
# generate Embeddings:
def get_embeddings(text):
  response = ollama.embeddings(
      model="mxbai-embed-large",
      prompt=text
  )
  return response['embedding']

In [26]:
# Store embeddings in Vectore DB:
for i, chunk in enumerate(chunks):
  embedding = get_embeddings(chunk)

  collaction.add(
      ids=[f"chunk_{i}"],
      embeddings=[embedding],
      documents=[chunk],
      metadatas=[{"source" : "uploaded_file", "chunk_id" : i}]
  )
  print("All chunks strored Successfully.")

All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks strored Successfully.
All chunks

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="rag_collection"
)

print("Chroma collection created.")